# FF-HEDM from raw CBF — auto-seed calibration + c-omp reconstruction

End-to-end far-field HEDM on a real dataset, using the **new MIDAS Python
stack**:

1. **Calibrate** the detector from a LaB6 (or any) calibrant with
   `midas-calibrate-v2`'s **auto-seed** one-shot path — no prior parameter
   file needed. Beam center and sample-to-detector distance are found from the
   rings themselves.
2. **Build** the FF parameter file from the refined geometry.
3. **Reconstruct** grains with `midas-pipeline` in FF mode using the
   **`c-omp` indexer backend** (the bundled unified C `midas_indexer`).
4. **Inspect** the recovered grains.

**Worked example:** a Ni sample on a Varex XRD-4343CT (2880x2880, 150 um pixel,
CBF), 1800 omega frames, calibrated against LaB6.

> ### ⚠️ Read detector images with the MIDAS reader, never `fabio`
> MIDAS works in a transposed + double-flipped frame
> (`pixels.reshape(nrows, ncols).T[::-1, ::-1]`). `fabio` returns the *raw*
> frame, which is a Y↔Z transpose + both-axes flip of the MIDAS frame — it
> gives a mirrored beam center and **mirrored tilts/distortion that are invalid
> for the pipeline**. Always read via `midas_zipper._read_cbf.read_cbf`
> (`midas-calibrate-v2` >= 0.2.2 already does this internally).


## Install — start from scratch

`midas-suite` is a meta-package that pulls the entire MIDAS Python stack
(calibration, zipper, peakfit, transforms, index **+ the c-omp `midas_indexer`
binary**, fit-grain, process-grains, the FF/PF orchestrators) in one command.

The `c-omp` indexer is a compiled OpenMP binary built at install time, so you
need a C toolchain + OpenMP (Linux: `gcc`; macOS: `brew install libomp`).


In [ ]:
# Run once in a fresh environment (uncomment to execute):
# %pip install "midas-suite[ff]"        # FF-HEDM stack + orchestrator
# Verify the c-omp indexer binary built:
from midas_index import backend_c
print("c-omp indexer available:", backend_c.available(), "->", backend_c.binary_path())
import midas_suite; print("installed:", len(midas_suite.installed()), "midas packages")


## 0. Configuration — edit this cell for your dataset

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # diplib/OpenMP on the seed step
from pathlib import Path

# ----- paths -----------------------------------------------------------------
WORK      = Path("/gdata/dm/MPE/OrthrosJr/analysis/sharma_work/indrajeet_ni_ff")
LAB6_DIR  = WORK / "LAB6/deg0_exp0_3/Varex_1"   # dir of calibrant CBF frames
NI_RAW    = WORK / "Ni_ht/Varex_1"              # dir of sample CBF frames
PS_TEMPLATE = WORK / "ps_ni.txt"                # FF thresholds (RingThresh, etc.)
CALIB_OUT = WORK / "nb_calib"
PS_V2     = WORK / "nb_ps_ni_v2.txt"
RECON_OUT = WORK / "nb_ni_recon"

# ----- calibrant (LaB6) ------------------------------------------------------
CAL_A, CAL_SG = 4.1569, 221         # LaB6 cubic, Pm-3m
WAVELENGTH    = 0.2066              # Angstrom
PX            = 150.0              # um
NPIX          = 2880
LSD_GUESS     = 960000.0           # um — rough nominal distance (order of magnitude)

# ----- sample / scan ---------------------------------------------------------
NUM_FILES_PER_SCAN = 1800          # 1 frame per CBF -> number of CBF files
                                   # (CRITICAL for per-frame CBF; default 1
                                   #  reads only the first file!)

# ----- compute ---------------------------------------------------------------
N_CPUS          = 64
INDEXER_BACKEND = "c-omp"          # bundled unified C midas_indexer
REFINE_BACKEND  = "c-omp"           # bundled unified C midas_fitgrain (FF refines
                                   # position; PF fixes it). "python" = PyTorch refiner.


## 1. Calibrate from the LaB6 rings (auto-seed, one-shot)
We median the calibrant frames (stationary powder rings ⇒ median removes zingers/hot pixels), then call `calibrate(...)`, which auto-seeds beam center + Lsd from the rings and LM-refines geometry + distortion.

In [ ]:
import numpy as np
from midas_zipper._read_cbf import read_cbf      # MIDAS reader (NOT fabio)
import glob

def median_calibrant(raw_dir):
    files = sorted(glob.glob(str(Path(raw_dir) / "*.cbf")))
    stack = np.stack([read_cbf(f, check_md5=False)[1].astype(np.float32)
                      for f in files])
    print(f"median over {len(files)} frames -> {stack.shape}")
    return np.median(stack, axis=0).astype(np.float64)

cal_img = median_calibrant(LAB6_DIR)

from midas_calibrate_v2 import calibrate
res = calibrate(
    cal_img,
    wavelength=WAVELENGTH, pxY=PX,
    calibrant={"a": CAL_A, "sg": CAL_SG},
    output_dir=str(CALIB_OUT),
    initial_Lsd=LSD_GUESS,
    verbose=True,
)
print(f"\nseed   BC=({res.seed_BC_y:.2f},{res.seed_BC_z:.2f})  "
      f"Lsd={res.seed_Lsd/1000:.2f} mm")
print(f"refine BC=({res.BC_y:.3f},{res.BC_z:.3f})  Lsd={res.Lsd/1000:.3f} mm  "
      f"ty={res.ty:+.3f} tz={res.tz:+.3f}")
print(f"strain: in-loop {res.in_loop_strain_uE:.1f} ue, "
      f"post-residual {res.post_residual_strain_uE:.1f} ue")


### Sanity check — predicted rings must sit on the measured signal
Uses `midas_hkls` (space-group extinction rules) to predict the allowed LaB6 rings at the refined geometry and overlays them on the median image.

In [ ]:
import math, matplotlib.pyplot as plt
from midas_hkls import SpaceGroup, Lattice, generate_hkls

refs = generate_hkls(SpaceGroup.from_number(CAL_SG),
                     Lattice(a=CAL_A, b=CAL_A, c=CAL_A,
                             alpha=90., beta=90., gamma=90.),
                     wavelength_A=res.wavelength_A, two_theta_max_deg=18.0)
ring_R = sorted(res.Lsd * math.tan(math.radians(r.two_theta_deg)) / res.pxY
                for r in refs)

fig, ax = plt.subplots(figsize=(9, 9))
pos = cal_img[cal_img > 0]
ax.imshow(np.log1p(cal_img.clip(min=1)), origin="lower", cmap="gray",
          vmin=np.log1p(np.percentile(pos, 60)),
          vmax=np.log1p(np.percentile(pos, 99.9)))
th = np.linspace(0, 2*np.pi, 500)
for r in ring_R:
    ax.plot(res.BC_y + r*np.cos(th), res.BC_z + r*np.sin(th),
            "-", color="lime", lw=0.7, alpha=0.85)
ax.plot(res.BC_y, res.BC_z, "ro", ms=7, mec="yellow")
ax.set_title(f"LaB6 median + predicted rings @ refined geom "
             f"(strain={res.post_residual_strain_uE:.0f} ue)")
ax.set_xlim(0, res.NrPixelsY); ax.set_ylim(0, res.NrPixelsZ)
plt.tight_layout(); plt.show()


## 2. Build the FF parameter file

Carry the **full** detector model from the powder calibration straight into
HEDM — refined geometry (`Lsd`, beam center, **all** tilts `tx/ty/tz`), the
**complete v2 distortion** (`iso_R2/4/6`, `a1..a6`, `phi1..phi6`), and the
Stage-4 spline residual map. **Nothing is zeroed, and there are no `p0..p14`** —
the `midas-calibrate-v2` result is written in its native v2 harmonic naming and
flows through the zipper → zarr → peakfit/transforms unchanged. Indexing/
refinement thresholds (`RingThresh`, `MinNrSpots`, …) are carried verbatim from
the template `ps` file.

`ff_paramstest_from_auto_result` does this in one call — the v2 result drives
everything; you never touch a coefficient by hand.

In [ ]:
from midas_calibrate_v2.compat.to_v1 import ff_paramstest_from_auto_result

# The full v2 calibration result → FF paramstest: geometry + ALL tilts + the
# complete v2 distortion harmonics + the spline residual map. No p0..p14.
ff_paramstest_from_auto_result(
    res, template_ps=PS_TEMPLATE, out_path=PS_V2,
    raw_folder=str(NI_RAW), n_pixels_y=NPIX, n_pixels_z=NPIX,
)
print(f"wrote {PS_V2}")
print(f"  Lsd={res.Lsd/1000:.3f} mm  BC=({res.BC_y:.2f},{res.BC_z:.2f})  "
      f"tx={res.tx:+.4f} ty={res.ty:+.4f} tz={res.tz:+.4f}")
print("  distortion (v2): "
      + ", ".join(f"{k}={v:+.2e}" for k, v in res.distortion.items()))
print(f"  residual map: {res.residual_corr_bin_path}")

## 3. Reconstruct with `midas-pipeline` (FF mode, c-omp index + refine)
One command runs the full FF pathway: convert (CBF→`.MIDAS.zip`) → hkl → peakfit → transforms → binning → **index (c-omp)** → **refine (c-omp)** → process-grains → `Grains.csv`. Run it on a multicore node.

Both backends are the bundled unified C binaries (no NLopt dependency): `midas_indexer` and `midas_fitgrain` (`FitUnified`). The refiner refines grain position in FF (validated to ~10µm median vs the legacy `FitPosOrStrainsOMP`) and fixes it to the voxel grid in PF. Set `REFINE_BACKEND='python'` for the differentiable PyTorch refiner instead (GPU/MPS, UQ).

> `--num-files-per-scan` **must** equal the number of single-frame CBF files — the default (1) reads only the first frame.
> Use `--device cpu` if the node has no GPU. The c-omp backends need `midas-index` / `midas-fit-grain` installed with an OpenMP toolchain.


In [ ]:
import subprocess, shlex
cmd = (f"midas-pipeline run --scan-mode ff --indexer-backend {INDEXER_BACKEND} --refine-backend {REFINE_BACKEND} "
       f"--num-files-per-scan {NUM_FILES_PER_SCAN} "
       f"--params {PS_V2} --result {RECON_OUT} --n-cpus {N_CPUS} "
       f"--machine local --device cpu")
print(cmd)
# Streams to the notebook. For very long runs, launch in a terminal with
# `nohup ... > log 2>&1 &` and watch the log instead.
env = {**os.environ, "KMP_DUPLICATE_LIB_OK": "TRUE"}
proc = subprocess.run(shlex.split(cmd), env=env)
print("return code:", proc.returncode)


## 4. Inspect the recovered grains

In [ ]:
import pandas as pd
gfile = RECON_OUT / "LayerNr_1" / "Grains.csv"
# Grains.csv has a multi-line % header; the data header row starts with %GrainID
lines = gfile.read_text().splitlines()
hdr = next(i for i, l in enumerate(lines) if l.startswith("%GrainID"))
ngrains = int([l for l in lines if l.startswith("%NumGrains")][0].split()[1])
cols = lines[hdr].lstrip("%").split()
df = pd.read_csv(gfile, sep="\t", skiprows=hdr+1, names=cols, comment="%")
print(f"NumGrains = {ngrains};  rows parsed = {len(df)}")
df[["GrainID","X","Y","Z","Confidence","GrainRadius","RMSErrorStrain"]].head()


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
sc = ax[0].scatter(df.X, df.Y, c=df.Confidence, s=8, cmap="viridis")
ax[0].set(xlabel="X (um)", ylabel="Y (um)", title="grain map (color=confidence)")
ax[0].set_aspect("equal"); fig.colorbar(sc, ax=ax[0])
ax[1].hist(df.Confidence, bins=40, color="steelblue")
ax[1].set(xlabel="completeness/confidence", ylabel="grains", title="confidence")
ax[2].hist(df.GrainRadius, bins=40, color="indianred")
ax[2].set(xlabel="grain radius (um)", ylabel="grains", title="grain size")
plt.tight_layout(); plt.show()


### (optional) Compare against a legacy reconstruction
If you have an older `Grains.csv`, compare grain counts and confidence distributions. Note: a calibration with a wrong beam center (e.g. from a fabio-mirrored read) will shift spot positions and degrade indexing — so a grain-count change after re-calibrating is expected and usually an improvement.

In [ ]:
LEGACY = WORK / "Ni_ht/ff_Al6/LayerNr_1/Grains.csv"
if LEGACY.exists():
    ll = LEGACY.read_text().splitlines()
    legacy_n = int([l for l in ll if l.startswith("%NumGrains")][0].split()[1])
    print(f"legacy NumGrains = {legacy_n};  new NumGrains = {ngrains}")
else:
    print("no legacy Grains.csv found — skipping comparison")


## 5. Refine `tx` from the grains, then re-reconstruct (two-pass)

Powder/calibrant calibration is **structurally blind to `tx`** — the in-plane
detector rotation about the beam. Symmetric Debye–Scherrer rings are invariant
under it, so the geometry from step 1 has `tx = 0`. Single-crystal **grain
spots** are *not* invariant: `tx` rotates each spot's azimuth η by ≈`tx` (with
near-zero effect on the radius), so the recovered grains pin it. (Verified on
real Ni data: `tx`→η is ~1:1, ΔR ≈ 0.)

Standard iterative detector calibration is therefore a **two-pass** flow:

1. Reconstruct with the powder geometry (`tx = 0`) — done above.
2. Refine `tx` from the grains and **re-reconstruct** with the corrected geometry.

The pipeline can do step 2's refine automatically inside an FF run with
`--grain-geometry-run` (the FF-only `grain_geometry` stage, after
`process_grains`). Below we use the standalone `midas-joint-ff-calibrate
grain-tx` tool on the existing layer dir — it's fast (no image re-processing) and
makes the before/after explicit.

In [ ]:
# Refine tx from the grains recovered in step 3 (standalone tool, η-sensitive
# 3D loss). Writes a corrected per-layer paramstest; we then fold the refined
# tx into the master FF param file for the re-run.
import subprocess, shlex, re
LAYER1 = RECON_OUT / "LayerNr_1"
PS_TX_LAYER = LAYER1 / "paramstest_graintx.txt"
cmd = (f"midas-joint-ff-calibrate grain-tx "
       f"--paramstest {LAYER1/'paramstest.txt'} --layer-dir {LAYER1} "
       f"--refine tx --kind angular --max-grains 50 --out {PS_TX_LAYER}")
print(cmd)
env = {**os.environ, "KMP_DUPLICATE_LIB_OK": "TRUE"}
subprocess.run(shlex.split(cmd), env=env)

def _tx(p):
    for line in open(p):
        s = line.split('#', 1)[0].strip().rstrip(';').split()
        if s and s[0] == 'tx':
            return float(s[1])
    return 0.0
tx_ref = _tx(PS_TX_LAYER)
print(f"refined tx = {tx_ref:+.6f} deg  (powder pass had tx = 0)")

# Corrected MASTER FF param file for the re-run: PS_V2 with tx set.
txt = PS_V2.read_text()
if re.search(r'(?m)^tx\b', txt):
    txt = re.sub(r'(?m)^tx\b.*$', f'tx {tx_ref:.6f}', txt)
else:
    txt += f"\ntx {tx_ref:.6f}\n"
PS_V2_TX = PS_V2.with_name(PS_V2.stem + "_tx" + PS_V2.suffix)
PS_V2_TX.write_text(txt)
print("wrote corrected master params →", PS_V2_TX)

Re-reconstruct with the corrected `tx`. A fresh result dir gives a clean
side-by-side; peak-fitting is geometry-independent, so on the same dir you could
instead `midas-pipeline resume --from transforms` to reuse step 3's peakfit and
only re-run transforms → binning → index → refine → grains.

In [ ]:
# Pass 2 — re-reconstruct with the tx-corrected geometry (long; same runtime as
# step 3). Comment out if you only want the tx value.
RECON_OUT2 = RECON_OUT.parent / (RECON_OUT.name + "_tx")
cmd2 = (f"midas-pipeline run --scan-mode ff --indexer-backend {INDEXER_BACKEND} "
        f"--refine-backend {REFINE_BACKEND} --num-files-per-scan {NUM_FILES_PER_SCAN} "
        f"--params {PS_V2_TX} --result {RECON_OUT2} --n-cpus {N_CPUS} "
        f"--machine local --device cpu")
print(cmd2)
subprocess.run(shlex.split(cmd2), env=env)
print("return code recorded above")

In [ ]:
# Compare pass 1 (tx=0) vs pass 2 (tx refined): grain count + median strain RMS.
def _load_grains(layer):
    g = layer / "Grains.csv"
    lines = g.read_text().splitlines()
    hdr = next(i for i, l in enumerate(lines) if l.startswith("%GrainID"))
    cols = lines[hdr].lstrip("%").split()
    import pandas as pd
    n = int([l for l in lines if l.startswith("%NumGrains")][0].split()[1])
    return n, pd.read_csv(g, sep="\t", skiprows=hdr+1, names=cols, comment="%")

n1, g1 = _load_grains(RECON_OUT / "LayerNr_1")
n2, g2 = _load_grains(RECON_OUT2 / "LayerNr_1")
print(f"grains:        tx=0 → {n1}      tx refined → {n2}")
print(f"median strain RMS:  {g1['RMSErrorStrain'].median():.3e}  →  "
      f"{g2['RMSErrorStrain'].median():.3e}")
print(f"median Confidence:  {g1['Confidence'].median():.4f}  →  "
      f"{g2['Confidence'].median():.4f}")